# GBR baseline — stesso hold-out di Chemprop

Notebook per confrontare la baseline **Gradient Boosting Regressor** con Chemprop usando gli stessi split.

Il notebook:

1. carica `PolymerPrediction.biceranoPolymers`;
2. ordina deterministicamente i documenti per `_id`;
3. applica il preprocessing di base;
4. legge direttamente da Chemprop `train_sample_ids.csv`, `val_sample_ids.csv` e `test_sample_ids.csv`;
5. mantiene esclusivamente quei campioni e assegna gli stessi split;
6. verifica che tutti gli SMILES del cohort siano validi per RDKit;
7. costruisce la baseline con **RDKit + Morgan + MACCS**;
8. esegue **Mutual Information → permutation importance → tuning GBR**;
9. valuta il modello sullo stesso test set di Chemprop.


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from pymongo import MongoClient

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, AllChem, MACCSkeys

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 28
DB_NAME = "PolymerPrediction"
COLLECTION_NAME = "polyInfo_kaggle"
TARGET = "Tg"

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()

while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent

if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "results" / "gbr" / f"gbr_{COLLECTION_NAME}"

CHEMPROP_OUTPUT_DIR = ROOT / "results" / "chemprop" / f"chemprop_{COLLECTION_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline baseline
N_MI = 50
MIN_PREVALENCE = 0.02

PARAM_GRID = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
}

print("Output:", OUTPUT_DIR.resolve())

## 1. Caricamento e preprocessing

In [ ]:
load_dotenv(ROOT / ".env")

MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI non trovato. Controlla il file .env.")

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

# Usa lo stesso ordinamento deterministico di Chemprop.
docs = list(collection.find({}).sort("_id", 1))
df = pd.DataFrame(docs)

if df.empty:
    raise RuntimeError(f"La collection {DB_NAME}.{COLLECTION_NAME} è vuota.")

df["sample_id"] = df["_id"].astype(str)

# Espansione di properties, se presente.
if "properties" in df.columns:
    properties = pd.json_normalize(df["properties"])
    df = pd.concat(
        [df.drop(columns=["properties"]), properties],
        axis=1,
    )

required = {"sample_id", "smiles", TARGET}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Colonne mancanti nel dataset: {sorted(missing)}")

dataset = df[["sample_id", "smiles", TARGET]].copy()

dataset[TARGET] = pd.to_numeric(dataset[TARGET], errors="coerce")
dataset = dataset.dropna(subset=["smiles", TARGET]).reset_index(drop=True)

dataset["smiles"] = dataset["smiles"].astype(str).str.strip()
dataset = dataset[dataset["smiles"] != ""].reset_index(drop=True)

# Pre-filtro già presente nel notebook Chemprop.
dataset = dataset[
    ~dataset["smiles"].str.contains(r"\[\[", na=False)
].reset_index(drop=True)

assert dataset["sample_id"].is_unique, "sample_id duplicati: controllare il dataset."

print(f"Campioni dopo il preprocessing di base: {len(dataset)}")
display(dataset.head())


## 2. Allineamento a Chemprop

Gli ID salvati dal run Chemprop sono usati per ricreare gli split identici.

In [ ]:
split_frames = []

for split_name in ("train", "val", "test"):
    split_path = CHEMPROP_OUTPUT_DIR / f"{split_name}_sample_ids.csv"

    if not split_path.exists():
        raise FileNotFoundError(
            f"File Chemprop non trovato: {split_path.resolve()}\n"
            "Esegui prima il notebook Chemprop aggiornato, con il filtro RDKit, "
            "e assicurati che OUTPUT_DIR sia results_chemprop_biceranoPolymers."
        )

    split_ids = pd.read_csv(split_path)

    if "sample_id" not in split_ids.columns:
        raise KeyError(
            f"{split_path} non contiene la colonna 'sample_id'."
        )

    split_ids = split_ids[["sample_id"]].copy()
    split_ids["sample_id"] = split_ids["sample_id"].astype(str)
    split_ids["split"] = split_name
    split_frames.append(split_ids)

split_assignments = pd.concat(split_frames, ignore_index=True)

if split_assignments["sample_id"].duplicated().any():
    duplicated = split_assignments.loc[
        split_assignments["sample_id"].duplicated(keep=False),
        ["sample_id", "split"],
    ]
    raise RuntimeError(
        "Uno o più sample_id compaiono in più split Chemprop:\n"
        + duplicated.to_string(index=False)
    )

assert set(split_assignments["split"]) == {"train", "val", "test"}

# Verifica che tutti gli ID prodotti da Chemprop esistano ancora nella collection.
missing_ids = sorted(
    set(split_assignments["sample_id"]) - set(dataset["sample_id"])
)

if missing_ids:
    raise RuntimeError(
        f"{len(missing_ids)} sample_id del run Chemprop non sono presenti "
        "nel dataset GBR dopo il preprocessing di base. "
        f"Primi ID mancanti: {missing_ids[:10]}"
    )

# L inner join mantiene esattamente il cohort accettato da Chemprop.
# sort=False conserva l'ordine deterministico del dataset originale.
dataset = dataset.merge(
    split_assignments,
    on="sample_id",
    how="inner",
    validate="one_to_one",
    sort=False,
)

if len(dataset) != len(split_assignments):
    raise RuntimeError(
        f"Allineamento incompleto: Chemprop contiene {len(split_assignments)} campioni, "
        f"GBR ne ha recuperati {len(dataset)}."
    )

# Controllo finale: dopo l'allineamento tutti gli SMILES devono essere processabili da RDKit.
rdkit_valid = dataset["smiles"].apply(
    lambda s: Chem.MolFromSmiles(s) is not None
)

if not rdkit_valid.all():
    invalid = dataset.loc[
        ~rdkit_valid,
        ["sample_id", "smiles", TARGET, "split"],
    ]
    raise RuntimeError(
        "Il cohort Chemprop contiene ancora SMILES non validi per RDKit. "
        "Probabilmente i file degli split provengono da un run precedente al filtro RDKit.\n"
        + invalid.to_string(index=False)
    )

print("✓ Cohort GBR identico a quello di Chemprop")
print(dataset["split"].value_counts())
print()

print(f"N totale = {len(dataset)}")
print(f"N train  = {(dataset['split'] == 'train').sum()}")
print(f"N val    = {(dataset['split'] == 'val').sum()}")
print(f"N test   = {(dataset['split'] == 'test').sum()}")

print(f"{TARGET} media: {dataset[TARGET].mean():.2f} K")
print(f"{TARGET} sigma: {dataset[TARGET].std(ddof=1):.2f} K")

# Salva una copia locale degli ID per rendere il run GBR autonomamente verificabile.
for split_name in ("train", "val", "test"):
    dataset.loc[
        dataset["split"] == split_name,
        ["sample_id"],
    ].to_csv(
        OUTPUT_DIR / f"{split_name}_sample_ids.csv",
        index=False,
    )


## 3. Featurizzazione RDKit + Morgan + MACCS

In [ ]:
def bitvect_to_array(fp):
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.astype(float)


rdkit_desc = Descriptors._descList
rdkit_names = [name for name, _ in rdkit_desc]

rows = []

for row_idx, smiles in enumerate(dataset["smiles"]):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise RuntimeError(
            f"Errore inatteso: SMILES non convertibile da RDKit alla riga "
            f"{row_idx}: {smiles}"
        )

    desc_values = []
    for _, func in rdkit_desc:
        try:
            desc_values.append(float(func(mol)))
        except Exception:
            desc_values.append(np.nan)

    morgan = bitvect_to_array(
        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=1024,
        )
    )

    maccs = bitvect_to_array(
        MACCSkeys.GenMACCSKeys(mol)
    )

    rows.append(
        np.concatenate([desc_values, morgan, maccs])
    )


feature_names = (
    rdkit_names
    + [f"morgan_{i}" for i in range(1024)]
    + [f"maccs_{i}" for i in range(167)]
)

X = pd.DataFrame(rows, columns=feature_names)
X = X.replace([np.inf, -np.inf], np.nan)

y = dataset[TARGET].to_numpy(dtype=float)

print("Shape iniziale:", X.shape)

## 4. Pulizia feature usando solo il development set

In [ ]:
train_mask = dataset["split"].eq("train").to_numpy()
val_mask   = dataset["split"].eq("val").to_numpy()
test_mask  = dataset["split"].eq("test").to_numpy()
dev_mask   = ~test_mask

# Tiene le colonne che hanno almeno un valore nel development set.
valid_cols = X.columns[~X.loc[dev_mask].isna().all()]
X = X[valid_cols].copy()

# Calcola l imputazione solo sul development set.
medians = X.loc[dev_mask].median()
X = X.fillna(medians)

# Rimuove eventuali colonne ancora non finite.
finite_cols = np.isfinite(
    X.loc[dev_mask].to_numpy(dtype=float)
).all(axis=0)
X = X.loc[:, finite_cols]

# Rimuove le colonne costanti nel development set.
nunique = X.loc[dev_mask].nunique(dropna=False)
X = X.loc[:, nunique > 1]

binary_cols = []
continuous_cols = []

for col in X.columns:
    values = set(X.loc[dev_mask, col].unique())
    if values.issubset({0.0, 1.0}):
        binary_cols.append(col)
    else:
        continuous_cols.append(col)

# Fingerprint troppo rare/frequenti eliminate sulla base del development set.
prevalence = X.loc[dev_mask, binary_cols].mean()
binary_cols = prevalence[
    (prevalence >= MIN_PREVALENCE)
    & (prevalence <= 1 - MIN_PREVALENCE)
].index.tolist()

print("Feature dopo pulizia:", X.shape[1])
print("Continue:", len(continuous_cols))
print("Binarie dopo prevalence filter:", len(binary_cols))

## 5. Mutual Information + permutation importance

In [ ]:
# Mutual Information sui descrittori continui, usando solo il development set.
n_mi = min(N_MI, len(continuous_cols))

if n_mi > 0:
    mi = mutual_info_regression(
        X.loc[dev_mask, continuous_cols],
        y[dev_mask],
        random_state=RANDOM_STATE,
    )

    mi_order = np.argsort(mi)[::-1][:n_mi]
    mi_cols = [continuous_cols[i] for i in mi_order]
else:
    mi_cols = []

candidate_cols = mi_cols + binary_cols

if not candidate_cols:
    raise RuntimeError("Nessuna feature disponibile dopo il pre-filter.")

print(
    f"Candidate: {len(candidate_cols)} "
    f"({len(mi_cols)} continue + {len(binary_cols)} binarie)"
)

# Modello di prova: training su train e controllo su validation.
probe = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=1.0,
    random_state=RANDOM_STATE,
)

probe.fit(
    X.loc[train_mask, candidate_cols],
    y[train_mask],
)

perm = permutation_importance(
    probe,
    X.loc[val_mask, candidate_cols],
    y[val_mask],
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

importance = pd.Series(
    perm.importances_mean,
    index=candidate_cols,
).sort_values(ascending=False)

selected_cols = importance[importance > 0].index.tolist()

# Fallback per evitare una selezione vuota.
if not selected_cols:
    selected_cols = importance.head(
        min(20, len(importance))
    ).index.tolist()

importance.rename("permutation_importance").to_csv(
    OUTPUT_DIR / "permutation_importance.csv",
    header=True,
)

pd.Series(selected_cols, name="feature").to_csv(
    OUTPUT_DIR / "selected_features.csv",
    index=False,
)

print("Feature selezionate:", len(selected_cols))
display(importance.head(20).to_frame())

## 6. Tuning GBR sul development set

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = GridSearchCV(
    estimator=GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
    param_grid=PARAM_GRID,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

search.fit(
    X.loc[dev_mask, selected_cols],
    y[dev_mask],
)

gbr = search.best_estimator_

print("Best params:", search.best_params_)
print(f"Best internal-CV MAE: {-search.best_score_:.3f} K")

pd.DataFrame(search.cv_results_).to_csv(
    OUTPUT_DIR / "grid_search_results.csv",
    index=False,
)

## 7. Valutazione finale sullo stesso test set di Chemprop

In [ ]:
y_test = y[test_mask]
y_pred = gbr.predict(
    X.loc[test_mask, selected_cols]
)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

sigma = dataset[TARGET].std(ddof=1)
mae_sigma = mae / sigma

summary = pd.DataFrame([{
    "model": "GBR baseline same-holdout",
    "dataset": COLLECTION_NAME,
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "n_total": len(dataset),
    "n_dev": int(dev_mask.sum()),
    "n_test": int(test_mask.sum()),
    "n_features_selected": len(selected_cols),
    "MAE_K": mae,
    "RMSE_K": rmse,
    "R2": r2,
    "sigma_K": sigma,
    "MAE_sigma": mae_sigma,
    "best_params": str(search.best_params_),
}])

summary.to_csv(
    OUTPUT_DIR / "summary.csv",
    index=False,
)

predictions = dataset.loc[
    test_mask,
    ["sample_id", "smiles", TARGET]
].copy().reset_index(drop=True)

predictions["Tg_pred_GBR"] = y_pred
predictions["abs_error"] = np.abs(y_test - y_pred)

predictions.to_csv(
    OUTPUT_DIR / "test_predictions_with_targets.csv",
    index=False,
)

display(summary.round(4))

print(
    f"\nGBR same-holdout | "
    f"MAE = {mae:.2f} K | "
    f"RMSE = {rmse:.2f} K | "
    f"R² = {r2:.3f} | "
    f"MAE/σ = {mae_sigma:.3f}"
)

In [ ]:
# Confronto tra valori predetti e osservati
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test, y_pred, alpha=0.65)

lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())

ax.plot(
    [lo, hi],
    [lo, hi],
    "--",
    linewidth=1.8,
)

ax.set_xlabel("Tg sperimentale [K]")
ax.set_ylabel("Tg predetta [K]")
ax.set_title(
    "GBR baseline — Bicerano\n"
    f"MAE = {mae:.1f} K | R² = {r2:.3f} | MAE/σ = {mae_sigma:.3f}"
)

fig.tight_layout()

plot_path = OUTPUT_DIR / "predicted_vs_true.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")

plt.show()

print("Grafico salvato in:", plot_path.resolve())